<a href="https://colab.research.google.com/github/DanielBR0612/supervised-machine-learning/blob/main/atividade2_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Atividade Prática**
<font size=3>

- **Tema:** amostragem e fluxo de trabalho.
- **Prazo de entrega:** 30 de Abril.

**Envie** o notebook **executado** em formato **ipynb** pelo [formulário](https://docs.google.com/forms/d/e/1FAIpQLSeHK5qrS-_wttHK6XXKBK2-SXv1nxU3Xhk8x__eP1ZrHulHDw/viewform?usp=header).

---

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **1. Questão:**
<font size=3>

Com base no *dataset* de [Fraudes de cartões de crédito](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe os dados e **imprima na tela** a proporção de classes;
2. Faça um divisão estratificada de dados em treinamento, validação e teste;
3. Faça a busca do hiperparâmetro $k$, **em um laço** `for`, do modelo **k-NN**. Utilize os dados de validação para medir a performance do modelo a cada valor de $k$;
4. **Imprima na tela** o melhor valor de $k$ e **retreine** o modelo com este valor. Utilize os dados de treinamento + validação para o *fit* do modelo;
5. Faça a avaliação final do melhor modelo com os dados de teste.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv('/content/drive/MyDrive/dataset/credit_card_fraud.csv')

In [4]:
y = df["Class"]
X = df.drop(['Class', 'Time'], axis=1)

In [5]:
np.unique(y, return_counts=True)

(array([0, 1]), array([2843,  492]))

In [6]:
X_train, X_dev, y_train, y_dev = train_test_split(X, y, test_size=0.4, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_dev, y_dev, test_size=0.5, stratify=y_dev)

In [7]:
print(f"Proporção da classe 1 no treino: {y_train.mean():.5f}")
print(f"Proporção da classe 1 na validation: {y_val.mean():.5f}")
print(f"Proporção da classe 1 no teste: {y_test.mean():.5f}")

Proporção da classe 1 no treino: 0.14743
Proporção da classe 1 na validation: 0.14843
Proporção da classe 1 no teste: 0.14693


In [8]:
scores = []

for k in range(1, 21):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)

    y_pred_val = model.predict(X_val)

    acc = accuracy_score(y_val, y_pred_val)

    scores.append(acc)

best_k = np.argmax(scores) + 1
print(f"Melhor k (validação): {best_k}")

Melhor k (validação): 12


In [9]:
final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(np.concatenate([X_train, X_val]),
                np.concatenate([y_train, y_val]))

y_pred = final_model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"Acurácia no conjunto de teste: {acc:.2f}")

Acurácia no conjunto de teste: 0.85


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


### **2. Questão:**
<font size=3>

Com base no *dataset* de [Pinguins](https://www.kaggle.com/datasets/parulpandey/palmer-archipelago-antarctica-penguin-data?select=penguins_size.csv), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe os dados e **imprima na tela** todas as classes das variáveis categóricas **e** suas proporções;
2. Caso exista alguma **classe indefinida**, veja quantas amostras esta classe apresenta. Caso seja um valor pequeno de amostras, remova a classe indefinida;
3. Observe se o *dataframe* apresenta valores `NaN`. Caso existam, remova a linha correspondente do *dataframe*;
4. Defina o atributo `species` como variável alvo (`y`), e as demais como `X`;
5. Realize as transformação necessárias nos dados categóricos e numéricos com base nas classes [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) e [`ColumnTransformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html);
6. Utilize a abordagem correta de treinamento e avaliação do modelo de **Regressão Logística** com base no tamanho do *dataset* e a proporção de classes.
   

In [10]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

In [11]:
pinguins = pd.read_csv('/content/drive/MyDrive/dataset/penguins_size.csv')

In [12]:
variaveis_categoricas = ['species', 'island', 'sex']

for variavel in variaveis_categoricas:
    print(f"--- Proporções para a variável: {variavel.upper()} ---")

    proporcoes = pinguins[variavel].value_counts(normalize=True, dropna=False) * 100

    print(proporcoes.apply(lambda x: f"{x:.2f}%"))
    print("\n" + "-"*40 + "\n")

--- Proporções para a variável: SPECIES ---
species
Adelie       44.19%
Gentoo       36.05%
Chinstrap    19.77%
Name: proportion, dtype: object

----------------------------------------

--- Proporções para a variável: ISLAND ---
island
Biscoe       48.84%
Dream        36.05%
Torgersen    15.12%
Name: proportion, dtype: object

----------------------------------------

--- Proporções para a variável: SEX ---
sex
MALE      48.84%
FEMALE    47.97%
NaN        2.91%
.          0.29%
Name: proportion, dtype: object

----------------------------------------



In [13]:
pinguins = pinguins[pinguins['sex'] != '.']
pinguins = pinguins.dropna(subset=['sex'])

In [14]:
variaveis_categoricas = ['species', 'island', 'sex']

for variavel in variaveis_categoricas:
    print(f"--- Proporções para a variável: {variavel.upper()} ---")

    proporcoes = pinguins[variavel].value_counts(normalize=True, dropna=False) * 100

    print(proporcoes.apply(lambda x: f"{x:.2f}%"))
    print("\n" + "-"*40 + "\n")

--- Proporções para a variável: SPECIES ---
species
Adelie       43.84%
Gentoo       35.74%
Chinstrap    20.42%
Name: proportion, dtype: object

----------------------------------------

--- Proporções para a variável: ISLAND ---
island
Biscoe       48.95%
Dream        36.94%
Torgersen    14.11%
Name: proportion, dtype: object

----------------------------------------

--- Proporções para a variável: SEX ---
sex
MALE      50.45%
FEMALE    49.55%
Name: proportion, dtype: object

----------------------------------------



In [15]:
y_pinguim = pinguins["species"]
X_pinguim = pinguins.drop(['species'], axis=1)

X_train_pinguim, X_dev_pinguim, y_train_pinguim, y_dev_pinguim = train_test_split(X_pinguim, y_pinguim, test_size=0.4, stratify=y_pinguim, random_state=42)
X_val_pinguim, X_test_pinguim, y_val_pinguim, y_test_pinguim = train_test_split(X_dev_pinguim, y_dev_pinguim, test_size=0.5, stratify=y_dev_pinguim, random_state=42)

In [16]:
le = LabelEncoder()

y_train_pinguim_encoded = le.fit_transform(y_train_pinguim)
y_test_pinguim_encoded = le.transform(y_test_pinguim)

print("\nAlvo de treino depois da codificação:")
print(f"Classes aprendidas pelo LabelEncoder: {le.classes_}")
print("Alvos de treino depois da codificação:", y_train_pinguim_encoded[:10])

print(f"Proporção da classe 1 no treino: {y_train_pinguim_encoded.mean():.5f}")
print(f"Proporção da classe 1 no teste: {y_test_pinguim_encoded.mean():.5f}")


Alvo de treino depois da codificação:
Classes aprendidas pelo LabelEncoder: ['Adelie' 'Chinstrap' 'Gentoo']
Alvos de treino depois da codificação: [2 0 0 0 1 0 1 1 1 2]
Proporção da classe 1 no treino: 0.91960
Proporção da classe 1 no teste: 0.91045


In [17]:
numerical_atributes = ['culmen_length_mm', 'culmen_depth_mm', 'flipper_length_mm', 'body_mass_g']
categorical_atributes = ['island', 'sex']

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()
preprocessor = ColumnTransformer(transformers=[('num', numeric_transformer, numerical_atributes),
                                               ('cat', categorical_transformer, categorical_atributes)],
                                 remainder='passthrough'
                                )

preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('num', StandardScaler(),
                                 ['culmen_length_mm', 'culmen_depth_mm',
                                  'flipper_length_mm', 'body_mass_g']),
                                ('cat', OneHotEncoder(), ['island', 'sex'])])

In [18]:
X_train_pinguim_processed = preprocessor.fit_transform(X_train_pinguim)
X_test_pinguim_processed = preprocessor.transform(X_test_pinguim)

print(f"X-train:{X_train_pinguim_processed.shape}, X-test:{X_test_pinguim_processed.shape}\n")
print("Amostra dos atributos de treino após o processamento:")
print(X_train_pinguim_processed[:5])

X-train:(199, 9), X-test:(67, 9)

Amostra dos atributos de treino após o processamento:
[[ 0.26532406 -1.38411153  0.76768538  0.64534289  1.          0.
   0.          1.          0.        ]
 [-1.18148815  0.94667525 -0.51190933 -0.59217323  1.          0.
   0.          0.          1.        ]
 [-0.90677697  0.74399814 -1.43606107 -0.59217323  0.          0.
   1.          0.          1.        ]
 [-0.94340513  0.79466742 -0.79626371 -0.77780065  0.          1.
   0.          1.          0.        ]
 [ 1.45573918  0.89600597 -0.29864355 -0.09716678  0.          1.
   0.          0.          1.        ]]


In [19]:
pipe = Pipeline(steps=[('preprocessor', preprocessor),
                       ('classifier', LogisticRegression())])

pipe

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['culmen_length_mm',
                                                   'culmen_depth_mm',
                                                   'flipper_length_mm',
                                                   'body_mass_g']),
                                                 ('cat', OneHotEncoder(),
                                                  ['island', 'sex'])])),
                ('classifier', LogisticRegression())])

In [20]:
y_val_pinguim_encoded = le.transform(y_val_pinguim)

pipe.fit(X_train_pinguim, y_train_pinguim_encoded)

validacao_preds = pipe.predict(X_val_pinguim)

acuracia_val = accuracy_score(y_val_pinguim_encoded, validacao_preds)

print(f"Acurácia no conjunto de validação: {acuracia_val:.5f}")

X_train_val_pinguim = pd.concat([X_train_pinguim, X_val_pinguim])
y_train_val_pinguim_encoded = le.transform(pd.concat([y_train_pinguim, y_val_pinguim]))

pipe.fit(X_train_val_pinguim, y_train_val_pinguim_encoded)

test_preds = pipe.predict(X_test_pinguim)
acuracia_test = accuracy_score(y_test_pinguim_encoded, test_preds)

print(f"Acurácia no conjunto de teste final: {acuracia_test:.5f}")

Acurácia no conjunto de validação: 1.00000
Acurácia no conjunto de teste final: 0.98507


### **3. Questão:**
<font size=3>

Com base no *dataset* de [Cogumelos](https://www.kaggle.com/datasets/uciml/mushroom-classification), disponível no diretório $\text{dataset/}\,$, realize os seguintes passos:
1. Importe os dados e **imprima na tela** a proporção da classificação definidas aos cogumelos (`class`);
2. Defina o atributo `class` como variável alvo (`y`), e as demais como `X`;
3. Faça a divisão dos *dataset* entre **treinamento** e **teste**;
4. Realize as transformação necessárias nos dados categóricos com base nas classes [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) e [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html);
5. Defina um objeto `Pipeline` para encadear o pré-processamento de `X_train` junto ao treinamento do modelo **k-NN**;
6. Utilize a classe [`RandomizedSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.RandomizedSearchCV.html) para realizar a busca do melhor par de hiperparâmetros `n_neighbors` e `p` . Para esta busca, defina 20 iterações e 3 divisões para a validação cruzada;
7. Faça a previsão e avaliação do melhor modelo com os dados de teste .
   

In [41]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

In [21]:
cogumelos = pd.read_csv('/content/drive/MyDrive/dataset/mushrooms.csv')

In [22]:
cogumelos

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8119,e,k,s,n,f,n,a,c,b,y,...,s,o,o,p,o,o,p,b,c,l
8120,e,x,s,n,f,n,a,c,b,y,...,s,o,o,p,n,o,p,b,v,l
8121,e,f,s,n,f,n,a,c,b,n,...,s,o,o,p,o,o,p,b,c,l
8122,p,k,y,n,f,y,f,c,n,b,...,k,w,w,p,w,o,e,w,v,l


In [25]:
variavel = 'class'
print(f"--- Proporções para a variável: {variavel.upper()} ---")

proporcoes = cogumelos[variavel].value_counts(normalize=True, dropna=False) * 100

print(proporcoes.apply(lambda x: f"{x:.2f}%"))
print("\n" + "-"*40 + "\n")

--- Proporções para a variável: CLASS ---
class
e    51.80%
p    48.20%
Name: proportion, dtype: object

----------------------------------------



In [31]:
y_cogumelo = cogumelos['class']
X_cogumelo = cogumelos.drop(['class'], axis=1)

X_train_cogumelo, X_dev_cogumelo, y_train_cogumelo, y_dev_cogumelo = train_test_split(X_cogumelo, y_cogumelo, test_size=0.4, stratify=y_cogumelo, random_state=42)
X_val_cogumelo, X_test_cogumelo, y_val_cogumelo, y_test_cogumelo = train_test_split(X_dev_cogumelo, y_dev_cogumelo, test_size=0.5, stratify=y_dev_cogumelo, random_state=42)

In [33]:
le = LabelEncoder()

y_train_cogumelo_encoded = le.fit_transform(y_train_cogumelo)
y_test_cogumelo_encoded = le.transform(y_test_cogumelo)

print("\nAlvo de treino depois da codificação:")
print(f"Classes aprendidas pelo LabelEncoder: {le.classes_}")
print("Alvos de treino depois da codificação:", y_train_cogumelo_encoded[:10])

print(f"Proporção da classe 1 no treino: {y_train_cogumelo_encoded.mean():.5f}")
print(f"Proporção da classe 1 no teste: {y_test_cogumelo_encoded.mean():.5f}")


Alvo de treino depois da codificação:
Classes aprendidas pelo LabelEncoder: ['e' 'p']
Alvos de treino depois da codificação: [0 0 1 0 1 0 0 1 1 0]
Proporção da classe 1 no treino: 0.48195
Proporção da classe 1 no teste: 0.48185


In [38]:
categorical_attributes_cogumelo = X_cogumelo.columns.tolist()

preprocessor_cogumelo = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(), categorical_attributes_cogumelo)
])

In [40]:
pipe_cogumelo = Pipeline(steps=[('preprocessor', preprocessor_cogumelo),
                       ('classifier', KNeighborsClassifier())])

pipe_cogumelo

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['cap-shape', 'cap-surface',
                                                   'cap-color', 'bruises',
                                                   'odor', 'gill-attachment',
                                                   'gill-spacing', 'gill-size',
                                                   'gill-color', 'stalk-shape',
                                                   'stalk-root',
                                                   'stalk-surface-above-ring',
                                                   'stalk-surface-below-ring',
                                                   'stalk-color-above-ring',
                                                   'stalk-color-below-ring',
                                                   'veil-type', 'veil-color',
                                                   'ring-number', 'ring-type',
                                                   'spore-print-color',
                                                   'population',
                                                   'habitat'])])),
                ('classifier', KNeighborsClassifier())])

In [42]:
param_grid = {'classifier__n_neighbors': randint(3, 20),
              'classifier__p': [1, 2] }

param_grid

{'classifier__n_neighbors': <scipy.stats._distn_infrastructure.rv_discrete_frozen at 0x7e0409bf1880>,
 'classifier__p': [1, 2]}

In [43]:
random_search = RandomizedSearchCV(estimator=pipe_cogumelo,
                                   param_distributions=param_grid,
                                   n_iter=10,
                                   cv=5,
                                   scoring='accuracy',
                                   random_state=42,
                                   verbose=1)

random_search.fit(X_train_cogumelo, y_train_cogumelo_encoded)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


RandomizedSearchCV(cv=5,
                   estimator=Pipeline(steps=[('preprocessor',
                                              ColumnTransformer(transformers=[('cat',
                                                                               OneHotEncoder(),
                                                                               ['cap-shape',
                                                                                'cap-surface',
                                                                                'cap-color',
                                                                                'bruises',
                                                                                'odor',
                                                                                'gill-attachment',
                                                                                'gill-spacing',
                                                                                'gill-size',
                                                                                'gill-color',
                                                                                'stalk-shape',
                                                                                'stalk-root',
                                                                                'stalk-surface-above-ring',
                                                                                'stalk-surface-below-ring',
                                                                                'stalk-color-above-ring',
                                                                                'stalk-color-below-ring',
                                                                                'veil-type',
                                                                                'veil-color',
                                                                                'ring-number',
                                                                                'ring-type',
                                                                                'spore-print-color',
                                                                                'population',
                                                                                'habitat'])])),
                                             ('classifier',
                                              KNeighborsClassifier())]),
                   param_distributions={'classifier__n_neighbors': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7e0409bf1880>,
                                        'classifier__p': [1, 2]},
                   random_state=42, scoring='accuracy', verbose=1)

In [44]:
print(f"Melhores parâmetros encontrados: {random_search.best_params_}")
print(f"Melhor score de acurácia (validação cruzada): {random_search.best_score_:.2f}")

Melhores parâmetros encontrados: {'classifier__n_neighbors': 5, 'classifier__p': 2}
Melhor score de acurácia (validação cruzada): 1.00


In [46]:
y_cogumelo_pred = random_search.predict(X_test_cogumelo)

acc = accuracy_score(y_test_cogumelo_encoded, y_cogumelo_pred)

print(f"Acurácia do melhor modelo: {acc:.2f}")

Acurácia do melhor modelo: 1.00
